# Hiraoka & Jo (2025) on DANDI:000044: the other neurons

Paper: T. Hiraoka and H.-H. Jo, "Hierarchical organization of bursty trains in event
sequences", doi:10.1063/5.0296506, read from the arXiv preprint body under `paper/`.
Dataset: DANDI:000044 version 0.210812.1516, the version the paper cites (ref. 39); see the
header of `reproduce_hiraoka_2025.ipynb` for the choice of version and the claims table.
This notebook tests one claim, `fig2-other-neurons-similar`, which needs a fit of every
pyramidal unit and is kept apart for its run time (about 25 minutes on 4 processes).

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1"); os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import bursttree as bt

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 300)
print("dandiset", bt.DANDISET, "version", bt.DANDI_VERSION)

## Data check

The units tables of all eight sessions: counts by cell type, and spike trains that are not
sorted in time (the burst analysis sorts each train before taking IETs).

In [ ]:
assets = bt.list_assets()
trains, rows = [], []
for a in assets:
    ses = a["path"].split("ses-")[1].split("_")[0]
    U = bt.load_units(bt.open_nwb(a["url"]))
    uns = [int(U["id"][i]) for i, s in enumerate(U["spikes"]) if np.any(np.diff(s) < 0)]
    rows.append(dict(session=ses, units=len(U["id"]), excitatory=int((U["cell_type"] == "excitatory").sum()),
                     inhibitory=int((U["cell_type"] == "inhibitory").sum()), unsorted_unit_ids=uns))
    for i, s in enumerate(U["spikes"]):
        if U["cell_type"][i] == "excitatory":
            trains.append(dict(session=ses, row=i, id=int(U["id"][i]), n=len(s), unsorted=int(U["id"][i]) in uns, spikes_ms=s * 1e3))
print(pd.DataFrame(rows).to_string())
print("excitatory (pyramidal) units:", len(trains), "; of them with unsorted spike times:", sum(t["unsorted"] for t in trains))
print("spikes per excitatory unit: min", min(t["n"] for t in trains), "median", int(np.median([t["n"] for t in trains])), "max", max(t["n"] for t in trains))

## fig2-other-neurons-similar

> Other neurons of the recordings show patterns similar to the representative unit, with
> heavy-tailed burst-size and merging-number distributions across timescales.

Criterion: the set is (a) the pyramidal units of session Achilles-10252013, or (b) the pyramidal
units of every session. "Pyramidal" is read as the `cell_type` value `excitatory`, the only
label for principal cells in this version. Only units whose relevant range holds at least three
grid points are used, with all spikes of the session (O2) and each fit of O3, under both grids
of O4. A unit shows the pattern when the power law is supported (R > 0, p < 0.05) for burst sizes
at more than half of its relevant timescales and for merging numbers at more than half of its
pairs (ratios 3.3, 5.9 and 11). Reproduced when more than half of the units show the pattern
under one reading. The distribution of each unit's median beta is reported.

In [ ]:
t1 = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    res = list(ex.map(bt.unit_all_patterns, [t["spikes_ms"] for t in trains], chunksize=2))
print(f"{len(trains)} units fitted in {(time.time() - t1) / 60:.1f} min")
R = pd.DataFrame([dict(session=t["session"], id=t["id"], n_spikes=t["n"], unsorted=t["unsorted"], **r)
                  for t, rr in zip(trains, res) for r in rr])
R["set_a"] = R.session == "Achilles-10252013"

def summarise(d):
    used = d[d.used]
    return pd.Series(dict(units=len(d), used=len(used), pattern=int(used.pattern.sum()),
                          frac_pattern=round(used.pattern.mean(), 3) if len(used) else np.nan,
                          median_b_frac=round(used.b_frac.median(), 3), median_k_frac=round(used.k_frac.median(), 3),
                          beta_q25=round(used.median_beta.quantile(0.25), 3), beta_median=round(used.median_beta.median(), 3),
                          beta_q75=round(used.median_beta.quantile(0.75), 3)))

print("set (a), Achilles-10252013:")
print(R[R.set_a].groupby(["grid", "reading"]).apply(summarise).to_string())
print("\nset (b), all sessions:")
print(R.groupby(["grid", "reading"]).apply(summarise).to_string())
print("\nset (b) by session, fraction of used units showing the pattern:")
print(R[R.used].pivot_table(index="session", columns=["grid", "reading"], values="pattern", aggfunc="mean").round(3).to_string())
print("\nset (b) by session, median over units of the median beta:")
print(R[R.used].pivot_table(index="session", columns=["grid", "reading"], values="median_beta", aggfunc="median").round(3).to_string())
print("\nunits dropped (fewer than three grid points):")
print(R[~R.used].groupby(["grid"]).id.count().to_string())
print("\nwithout the units with unsorted spike times, set (b):")
print(R[~R.unsorted].groupby(["grid", "reading"]).apply(summarise)[["used", "pattern", "frac_pattern"]].to_string())

**Result: not reproduced under the discrete KS fit adopted here (an assumption); reproduced under
the continuous fits.** With the discrete KS fit, 129 and 130 of 554 and 562 units show the pattern
in set (b) (0.233 and 0.231), and 14 and 9 of 120 in set (a) (0.117 and 0.075), on grids (a) and
(b). With the continuous KS fit, the powerlaw package's default, set (b) passes narrowly (303 of
554, 0.547; 315 of 562, 0.560) and set (a) does not (0.400 and 0.392); with the continuous fit
from xmin = 1 both sets pass (0.883 to 0.906). The discrete fit from xmin = 1 gives 0.017 to
0.215. Under the discrete KS fit the median over units of the median beta is 2.404 and 2.400 in
set (b) (interquartile range 2.156 to 2.648 on grid (a)) and 2.550 and 2.538 in set (a). Leaving
out the two units with unsorted spike times changes no fraction by more than 0.003.

### Burst sizes and merging numbers taken apart

The pattern needs both parts. Their separate rates show which part fails where a reading
falls short.

In [ ]:
used = R[R.used]
part = used.assign(burst_ok=used.b_frac > 0.5, merge_ok=used.k_frac > 0.5).groupby(["grid", "reading"]).agg(
    used=("id", "size"), burst_ok=("burst_ok", "mean"), merge_ok=("merge_ok", "mean"), both=("pattern", "mean"))
print("set (b):"); print(part.round(3).to_string())
parta = used[used.set_a].assign(burst_ok=lambda d: d.b_frac > 0.5, merge_ok=lambda d: d.k_frac > 0.5).groupby(["grid", "reading"]).agg(
    used=("id", "size"), burst_ok=("burst_ok", "mean"), merge_ok=("merge_ok", "mean"), both=("pattern", "mean"))
print("\nset (a):"); print(parta.round(3).to_string())
print(f"\nnotebook time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (main process)")

**Result (parts).** Under the discrete KS fit, the burst-size part of the pattern holds in 0.736
and 0.769 of the units of set (b), and the merging-number part in 0.247 and 0.238: it is the
merging numbers that fail. In set (a), the merging part holds in 0.117 and 0.075.